In [ ]:
from qarp.operators import QubitOperator

from qarp.blocks import HnBlock, LinearEntanglingBlock, HEABlock, XnBlock, ComputationalBasisStateBlock, IdentityBlock, GHZLikeStateBlock, PauliBlock
from qarp.algorithms import SWAPTest, HadamardTest, MirrorTest, TermwiseHadamardTest, TermwiseSWAPTest, InterferometricTest, StateVector, PauliAveraging
from qarp.engines import QarpEngine
from qarp.plotting import plot

from copy import deepcopy

---

## How to sample a Block

<div style="background-color: #fcc651e8; border-left: 4px solid #ff9900ff; padding: 10px; margin: 10px 0; color: black;">
<strong>Note:</strong> Append <code>ReadoutBlock(n_qubits)</code> as the final child to make the <code>Measure</code> ops explicit in the IR (required for QIR/QASM emission; the QarpEngine itself reports counts over all qubits with or without them).
</div>

In [ ]:
from qarp.blocks import CompositeBlock, ReadoutBlock, HnBlock
from qarp.engines import QarpEngine
from qarp.algorithms import Sampler

n_qubits = 4

block = CompositeBlock([HnBlock(n_qubits=n_qubits), ReadoutBlock(n_qubits=n_qubits)])

primitive = Sampler(ket=block)

engine = QarpEngine()
engine.build([primitive])
results = engine.run()

print(results)

In [ ]:
import qarpx as qx
from qarp.blocks import CompositeBlock, ReadoutBlock, LayerBlock
from qarp.engines import QarpEngine
from sympy import Symbol

n_qubits = 4

thetas = [Symbol(f'theta_{i}') for i in range(4)]
block = CompositeBlock([LayerBlock(qx.GateType.Ry, 4, parameters=thetas), ReadoutBlock(n_qubits=4)]).build()

parameter_mapping = {thetas[0]: 0.1, thetas[1]: 0.4, thetas[2]: 0.3, thetas[3]: 0.4}
block_with_set_params = block.set_symbols(parameter_mapping) # NOTE: set_symbols returns a new Block instance with the parameters set (does not modify in place)

primitive = Sampler(ket=block_with_set_params)

engine = QarpEngine()
engine.build([primitive])
results = engine.run()

print(results)

---

## How to compute overlaps, expectation values, and transition amplitudes

- overlaps $\rightarrow \langle \text{bra} | \text{ket} \rangle $
- expectation values $\rightarrow \langle \text{ket} | \text{operator} | \text{ket} \rangle $
- transition amplitudes $\rightarrow \langle \text{bra} | \text{operator} | \text{ket} \rangle $

### Target

In [ ]:
op = PauliBlock("XYZX")
bra_block = HnBlock(n_qubits=4, name="$U_\\psi$")
ket_block = ComputationalBasisStateBlock([1, 1, 0, 1])

m0 = Sampler(ket=bra_block).build()
print(m0)
print(m0.target)

m1 = HadamardTest(bra=bra_block, ket=ket_block).build()
print(m1)
print(m1.target)

m2 = HadamardTest(bra=ket_block, operator=op, ket=ket_block).build()
print(m2)
print(m2.target)

m3 = HadamardTest(bra=bra_block, operator=op, ket=ket_block).build()
print(m3)
print(m3.target)

### Example overlap

NOTE: Computing expectation values and transition amplitudes follows the same 

In [ ]:
_tmpl = HEABlock(n_qubits=4, n_layers=1, real=True, linear=True, circular=True, use_cz=True)
hea_block = _tmpl.set_symbols({s: 0.1 for s in _tmpl.symbols}).build()

# Create separate instances for bra and ket
bra_block = deepcopy(hea_block)
ket_block = deepcopy(hea_block)

# Create the SwapTest measurement
m = SWAPTest(bra=bra_block, ket=ket_block)

# Initialize the Qarp engine and run the measurement
engine = QarpEngine()
engine.build([m])
engine.run()

### Pauli Averaging

In [ ]:
state_block = ComputationalBasisStateBlock([1, 1, 0, 1])

# Note that the operator is defined as qarp.operators.QubitOperator, not Block
op = QubitOperator("Z0 Z1") - QubitOperator("Z2 Z3") + QubitOperator("Z0 Y1 X3")

m = PauliAveraging(bra=state_block, operator=op, ket=state_block)

engine = QarpEngine()
engine.build([m])
print(f"number of measurement circuits: {len(m.sub_blocks)}")

m.sub_blocks[0].plot(spacing=0.5)
m.sub_blocks[1].plot(spacing=0.5)
print(f"Engine run result: {engine.run()}")

# Here, we show the result of manually constructing the measurement circuits for each term in the operator
m0 = StateVector(bra=state_block, operator=QubitOperator("Z0 Z1"), ket=state_block).build()
m1 = StateVector(bra=state_block, operator=QubitOperator("Z2 Z3"), ket=state_block).build()
m2 = StateVector(bra=state_block, operator=QubitOperator("Z0 Y1 X3"), ket=state_block).build()
engine = QarpEngine()
engine.build([m0, m1, m2])
print(f"Engine run result: {engine.run()}")

### Comparison HadamardTest - SWAPTest - StateVector

In [ ]:
_tmpl = HEABlock(n_qubits=4, n_layers=1, real=True, linear=True, circular=True, use_cz=True)
hea_block_1 = _tmpl.set_symbols({s: 0.1 for s in _tmpl.symbols}).build()
hea_block_2 = _tmpl.set_symbols({s: 0.2 for s in _tmpl.symbols}).build()
hea_block_3 = _tmpl.set_symbols({s: 0.3 for s in _tmpl.symbols}).build()

# Create separate instances for bra and ket
bra_block = HnBlock(n_qubits=4, name="$U_\\psi$")
ket_block = ComputationalBasisStateBlock([1, 1, 1, 1])

identity = IdentityBlock(4)

# Create the HadamardTest measurement
m1 = HadamardTest(bra=bra_block, ket=ket_block)

# Create the SwapTest measurement
s1 = SWAPTest(bra=bra_block, ket=ket_block)

# Create the StateVector measurement
sv1 = StateVector(bra=bra_block, ket=ket_block)

# Initialize the Qarp engine and run the measurement
engine = QarpEngine()
engine.build([m1, s1, sv1])
engine.run()

print("HadamardTest result:", abs(m1.result)**2)
print("SwapTest result:", s1.result)
print("StateVector result:", abs(sv1.result)**2)

m1.sub_blocks[0].plot()
s1.sub_blocks[0].plot()

---

## How to compute multiple quantities at the same time

### Multiple expectation values "manually"

In [ ]:
_tmpl = HEABlock(n_qubits=4, n_layers=1, real=True, linear=True, circular=True, use_cz=False)
hea_block_1 = _tmpl.set_symbols({s: 0.1 for s in _tmpl.symbols}).build()
hea_block_2 = _tmpl.set_symbols({s: 0.2 for s in _tmpl.symbols}).build()
hea_block_3 = _tmpl.set_symbols({s: 0.3 for s in _tmpl.symbols}).build()

# Create separate instances for bra and ket
bra_block = HnBlock(n_qubits=4, name="$U_\\psi$")
op1_block = hea_block_1
op2_block = hea_block_2
op3_block = hea_block_3
ket_block = LinearEntanglingBlock(n_qubits=4, circular=True, use_cz=False, name="$U_\\phi^2 \\;\\sum_i^k x_i$")

# Create the HadamardTest measurement
m1 = HadamardTest(bra=bra_block, operator=op1_block, ket=ket_block)
m2 = HadamardTest(bra=bra_block, operator=op2_block, ket=ket_block)
m3 = HadamardTest(bra=bra_block, operator=op3_block, ket=ket_block)

# Initialize the Qarp engine and run the measurement
engine = QarpEngine()
engine.build([m1, m2, m3])
engine.run()

### Multiple expectation values using Termwise algorithms

In [ ]:
_tmpl = HEABlock(n_qubits=4, n_layers=1, real=True, linear=True, circular=True, use_cz=False)
hea_block_1 = _tmpl.set_symbols({s: 0.1 for s in _tmpl.symbols}).build()
hea_block_2 = _tmpl.set_symbols({s: 0.2 for s in _tmpl.symbols}).build()
hea_block_3 = _tmpl.set_symbols({s: 0.3 for s in _tmpl.symbols}).build()

# Create separate instances for bra and ket
bra_block = HnBlock(n_qubits=4, name="$U_\\psi$")
ket_block = LinearEntanglingBlock(n_qubits=4, circular=True, use_cz=False, name="ket")

# Create the circuit blocks for the operators
op1_block = hea_block_1
op2_block = hea_block_2
op3_block = hea_block_3
op = [op1_block, op2_block, op3_block]

# Create the QubitOperator as an alternative
op1 = QubitOperator('X0 Y1') + 0.5 * QubitOperator('Z0 Z1')
op2 = 1/2 * QubitOperator('X0 Y1 Z3') + 2 * QubitOperator('Z0 Z1')

# Create the HadamardTest measurement
m1 = TermwiseHadamardTest(bra=bra_block, operator=op1, ket=ket_block,
                         real=True, imaginary=True)
m2 = TermwiseHadamardTest(bra=bra_block, operator=op2, ket=ket_block,
                         real=True, imaginary=True)

# Initialize the Qarp engine and run the measurement
engine = QarpEngine()
engine.build([m1, m2])
engine.run()

print("m1: ", m1.result_list)
print("m2: ", m2.result_list)

m1.sub_blocks[0].plot(spacing=0.4)
m1.sub_blocks[1].plot(spacing=0.4)

---

## Other tests

### Interferometric Test

In [ ]:
ComputationalBasisStateBlock([1, 0, 1, 1]).build().plot()
GHZLikeStateBlock([1, 0, 1, 1]).build().plot()
GHZLikeStateBlock([1, 0, 1, 1], dephase=True).build().plot()

#############################

# Example usage of InterferometricTest
# Here we compute the expectation value of the identity operator between the states |01> and |01>
# The expected result is 1.0 + 0.0j
#
ket = ComputationalBasisStateBlock([0, 1])
bra = ket
op = IdentityBlock(2)
m = InterferometricTest(bra=bra, operator=op, ket=ket, real=True, imaginary=True, sampling_algorithm=MirrorTest())

engine = QarpEngine()
engine.build([m])
engine.run()